# 🎮 Teamobi 2026 – Google Colab Server
Chạy private server Teamobi 2026 trên Google Colab + expose port qua ngrok TCP.

**Các bước:**
1. Chạy từng cell theo thứ tự
2. Lấy ngrok authtoken miễn phí tại https://dashboard.ngrok.com/get-started/your-authtoken
3. Điền token vào Cell 2 → chạy → lấy địa chỉ TCP để kết nối game

In [ ]:
# ── CELL 1: Cài packages ─────────────────────────────────
import subprocess, os

print('📦 Cài openjdk-17, MariaDB, unrar...')
cmds = [
    'apt-get update -qq',
    'apt-get install -y -qq openjdk-17-jdk mariadb-server unrar wget curl libssl-dev',
    'pip install -q gdown pyngrok'
]
for c in cmds:
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print(f'✅ {c.split()[0]}')
    else:
        print(f'⚠️  {c.split()[0]}: {r.stderr[-200:]}')

# Kiểm tra Java
v = subprocess.run('java -version', shell=True, capture_output=True, text=True)
print('\nJava:', v.stderr.strip().split('\n')[0])

In [ ]:
# ── CELL 2: Setup ngrok ──────────────────────────────────
# Lấy token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok, conf

NGROK_TOKEN = ''  # ← Dán authtoken của bạn vào đây

if not NGROK_TOKEN:
    raise ValueError('❌ Bạn chưa điền NGROK_TOKEN! Lấy tại https://dashboard.ngrok.com')

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token OK')

In [ ]:
# ── CELL 3: Khởi động MariaDB + tạo DB ──────────────────
import subprocess, time

DB_NAME = 'teamobi2026'
DB_PASS = 'teamobi@2026'
INSTALL_DIR = '/content/teamobi'
os.makedirs(INSTALL_DIR, exist_ok=True)

# Start MariaDB
subprocess.run('service mariadb start', shell=True, capture_output=True)
time.sleep(3)
r = subprocess.run('mysqladmin ping 2>&1', shell=True, capture_output=True, text=True)
print('MariaDB:', r.stdout.strip() or r.stderr.strip())

# Set root password + tạo DB
sql_init = f"""
ALTER USER 'root'@'localhost' IDENTIFIED BY '{DB_PASS}';
FLUSH PRIVILEGES;
CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4;
USE `{DB_NAME}`;
CREATE TABLE IF NOT EXISTS account (
  id INT AUTO_INCREMENT PRIMARY KEY,
  username VARCHAR(50) NOT NULL UNIQUE,
  password VARCHAR(64) NOT NULL,
  email VARCHAR(100) DEFAULT '',
  role INT DEFAULT 0 COMMENT '0=player,1=gm,2=admin',
  status INT DEFAULT 1,
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE IF NOT EXISTS nhan_vat (
  id INT AUTO_INCREMENT PRIMARY KEY,
  account_id INT,
  name VARCHAR(50),
  level INT DEFAULT 1,
  vang BIGINT DEFAULT 0,
  ngoc BIGINT DEFAULT 0,
  exp BIGINT DEFAULT 0,
  map_id INT DEFAULT 1,
  FOREIGN KEY (account_id) REFERENCES account(id)
);
CREATE TABLE IF NOT EXISTS meo_lu (
  id INT AUTO_INCREMENT PRIMARY KEY,
  account_id INT,
  pet_name VARCHAR(50) DEFAULT 'Meo Lu',
  pet_level INT DEFAULT 1,
  pet_exp INT DEFAULT 0,
  pet_skill VARCHAR(200) DEFAULT 'Lucky Strike',
  FOREIGN KEY (account_id) REFERENCES account(id)
);
"""

r2 = subprocess.run(
    f"mysql -u root --connect-expired-password -e \"{sql_init}\"",
    shell=True, capture_output=True, text=True
)
if r2.returncode == 0:
    print(f'✅ Database `{DB_NAME}` OK')
else:
    # Thử với password
    r3 = subprocess.run(
        f'mysql -u root -p{DB_PASS} -e "CREATE DATABASE IF NOT EXISTS `{DB_NAME}`;"',
        shell=True, capture_output=True, text=True
    )
    print('DB:', r3.stdout or r3.stderr)

In [ ]:
# ── CELL 4: Tải Teamobi2026.rar từ Google Drive ─────────
import gdown, os

DRIVE_ID = '1uH2O2FtuGpIQfIYVAhi9wcuxfDjTQddY'
RAR_PATH = '/content/teamobi/Teamobi2026.rar'

if os.path.exists(RAR_PATH) and os.path.getsize(RAR_PATH) > 100_000_000:
    print(f'⚠️  File đã tồn tại ({os.path.getsize(RAR_PATH)//1024//1024} MB), bỏ qua.')
else:
    print('📥 Đang tải (~630MB)...')
    url = f'https://drive.google.com/uc?id={DRIVE_ID}'
    gdown.download(url, RAR_PATH, quiet=False, fuzzy=True)
    size = os.path.getsize(RAR_PATH) // 1024 // 1024
    print(f'✅ Tải xong: {size} MB')

In [ ]:
# ── CELL 5: Giải nén + tìm JAR ──────────────────────────
import subprocess, glob

EXTRACT_DIR = '/content/teamobi'
print('📂 Giải nén RAR...')
r = subprocess.run(
    f'unrar x -o+ "{RAR_PATH}" "{EXTRACT_DIR}/"',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    subprocess.run(f'unrar e -o+ "{RAR_PATH}" "{EXTRACT_DIR}/"', shell=True)

# Tìm JAR
all_jars = glob.glob(f'{EXTRACT_DIR}/**/*.jar', recursive=True)
print(f'\n📦 Tìm thấy {len(all_jars)} file JAR:')
for j in all_jars:
    print(f'   {j}')

# Tự phân loại
import re
jar_game = next((j for j in all_jars if re.search(r'game|srcgame|server|main', os.path.basename(j), re.I)), all_jars[0] if all_jars else None)
jar_login = next((j for j in all_jars if re.search(r'login|auth', os.path.basename(j), re.I)), None)
sql_files = glob.glob(f'{EXTRACT_DIR}/**/*.sql', recursive=True)
apk_files = glob.glob(f'{EXTRACT_DIR}/**/*.apk', recursive=True)

print(f'\n🎮 JAR game  : {jar_game}')
print(f'🔑 JAR login : {jar_login}')
print(f'🗄️  SQL files : {sql_files}')
print(f'📱 APK files : {apk_files}')

# Import SQL nếu có
if sql_files:
    for sf in sql_files:
        r2 = subprocess.run(f'mysql -u root -p{DB_PASS} {DB_NAME} < "{sf}"', shell=True, capture_output=True, text=True)
        print(f'SQL import {os.path.basename(sf)}: {"OK" if r2.returncode==0 else r2.stderr[-100:]}')

In [ ]:
# ── CELL 6: Chạy Server + ngrok TCP ─────────────────────
import subprocess, time, threading
from pyngrok import ngrok

GAME_PORT = 14445
LOG_DIR = '/content/teamobi/logs'
os.makedirs(LOG_DIR, exist_ok=True)

procs = []

# Start Login Server (nếu có)
if jar_login:
    print(f'🔑 Login Server: {os.path.basename(jar_login)}')
    lp = subprocess.Popen(
        ['java', '-Xms128m', '-Xmx256m', '-jar', jar_login],
        cwd=os.path.dirname(jar_login),
        stdout=open(f'{LOG_DIR}/login.log', 'w'),
        stderr=subprocess.STDOUT
    )
    procs.append(('login', lp))
    time.sleep(3)
    print(f'   PID: {lp.pid}')

# Start Game Server
print(f'\n🎮 Game Server: {os.path.basename(jar_game)}')
gp = subprocess.Popen(
    ['java', '-Xms256m', '-Xmx512m', '-jar', jar_game],
    cwd=os.path.dirname(jar_game),
    stdout=open(f'{LOG_DIR}/game.log', 'w'),
    stderr=subprocess.STDOUT
)
procs.append(('game', gp))
time.sleep(5)
print(f'   PID: {gp.pid}')

# Mở ngrok TCP tunnel
print(f'\n🌐 Mở ngrok TCP tunnel port {GAME_PORT}...')
tunnel = ngrok.connect(GAME_PORT, 'tcp')
tcp_addr = tunnel.public_url.replace('tcp://', '')
host, port = tcp_addr.split(':')

print()
print('=' * 55)
print('  🎮  SERVER TEAMOBI 2026 ĐANG CHẠY!')
print('=' * 55)
print(f'  📡 Địa chỉ : {host}')
print(f'  🔌 Port    : {port}')
print(f'  📱 Kết nối game: {host}:{port}')
print('=' * 55)
print('  ⚠️  Colab tự ngắt sau ~12h. Giữ tab mở!')
print('='*55)

In [ ]:
# ── CELL 7: Quản lý tài khoản (tuỳ chọn) ───────────────
import subprocess, hashlib

def db_exec(sql):
    r = subprocess.run(
        f'mysql -u root -p{DB_PASS} {DB_NAME} -e "{sql}"',
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip()

def md5(s): return hashlib.md5(s.encode()).hexdigest()

def add_account(username, password, role=0):
    h = md5(password)
    result = db_exec(f"INSERT IGNORE INTO account (username,password,role) VALUES ('{username}','{h}',{role});")
    print(f'✅ Tài khoản [{username}] role={role} đã tạo!')

def list_accounts():
    r = db_exec('SELECT id,username,role,status,created_at FROM account ORDER BY id DESC LIMIT 20;')
    print(r or '(Chưa có tài khoản)')

def add_gold(char_name, vang=0, ngoc=0):
    db_exec(f"UPDATE nhan_vat SET vang=vang+{vang}, ngoc=ngoc+{ngoc} WHERE name='{char_name}';")
    print(f'✅ +{vang} vàng +{ngoc} ngọc cho [{char_name}]')

# ── Ví dụ sử dụng (bỏ comment để chạy) ──
# add_account('admin', 'admin123', role=2)    # Tạo admin
# add_account('player1', 'pass123', role=0)   # Tạo player
# list_accounts()                              # Xem danh sách
# add_gold('TenNhanVat', vang=100000, ngoc=500)  # Thêm vàng ngọc

print('✅ Hàm quản lý sẵn sàng. Bỏ comment ở trên để dùng.')

In [ ]:
# ── CELL 8: Xem log realtime ─────────────────────────────
import time

# Xem 50 dòng cuối của game.log
with open(f'{LOG_DIR}/game.log', 'r', errors='ignore') as f:
    lines = f.readlines()
    print(''.join(lines[-50:]) if lines else '(Chưa có log)')

# Hoặc theo dõi realtime (Ctrl+C để dừng):
# import subprocess
# subprocess.run(['tail', '-f', f'{LOG_DIR}/game.log'])